# Task 3 — Password Cracking using CUDA
**6CS005 High Performance Computing**

⚠️ **Requires a GPU runtime in Google Colab.**  
Go to **Runtime → Change runtime type → GPU** before running.

This notebook:
1. Verifies GPU availability
2. Writes the CUDA source inline
3. Generates an encrypted password file using Python
4. Compiles and runs the CUDA programme
5. Displays and verifies results

## Step 1 — Verify GPU

In [ ]:
!nvidia-smi
!nvcc --version

## Step 2 — Write the CUDA source file

In [ ]:
%%writefile password_crack.cu
/*
 * Task 3: Password Cracking using CUDA
 * 6CS005 High Performance Computing
 * Compile: nvcc -O2 -o password_crack password_crack.cu
 * Usage:   ./password_crack <encrypted_file>
 *
 * Encryption: Caesar +3 then XOR with (position+1)
 * Decryption: XOR with (position+1) then Caesar -3
 */
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <cuda_runtime.h>

#define MAX_PASS_LEN   128
#define THREADS_PER_BLOCK 256

#define CUDA_CHECK(call) do { \
    cudaError_t _e=(call); \
    if(_e!=cudaSuccess){fprintf(stderr,"[CUDA ERR] %s at %s:%d\n",cudaGetErrorString(_e),__FILE__,__LINE__);exit(1);} \
} while(0)

/* ── Decryption kernel: one thread per password ── */
__global__ void decrypt_kernel(const char *d_enc, char *d_dec, int n) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if(idx >= n) return;
    const char *enc = d_enc + idx * MAX_PASS_LEN;
    char       *dec = d_dec + idx * MAX_PASS_LEN;
    int len=0;
    while(len < MAX_PASS_LEN-1 && enc[len]!='\0') len++;
    for(int i=0;i<len;i++){
        unsigned char c=(unsigned char)enc[i];
        c = c ^ (unsigned char)(i+1);   /* undo XOR   */
        c = (c - 3) & 0xFF;            /* undo +3    */
        dec[i]=(char)c;
    }
    dec[len]='\0';
}

static int read_passwords(const char *fname, char **buf, int *n){
    FILE *fp=fopen(fname,"r"); if(!fp){perror("fopen");return -1;}
    int cnt=0; char tmp[MAX_PASS_LEN*2];
    while(fgets(tmp,sizeof(tmp),fp)) cnt++;
    rewind(fp);
    if(cnt==0){fprintf(stderr,"[ERROR] Empty file\n");fclose(fp);return -1;}
    char *b=(char*)calloc((size_t)cnt*MAX_PASS_LEN,1); if(!b){perror("calloc");fclose(fp);return -1;}
    int idx=0;
    while(fgets(tmp,sizeof(tmp),fp)&&idx<cnt){
        size_t l=strlen(tmp);
        while(l>0&&(tmp[l-1]=='\n'||tmp[l-1]=='\r'))tmp[--l]='\0';
        if(l==0){cnt--;continue;}
        strncpy(b+idx*MAX_PASS_LEN,tmp,(l<(size_t)(MAX_PASS_LEN-1))?l:(size_t)(MAX_PASS_LEN-1));
        idx++;
    }
    cnt=idx; fclose(fp); *buf=b; *n=cnt; return 0;
}

int main(int argc,char*argv[]){
    if(argc<2){fprintf(stderr,"Usage: %s <encrypted_file>\n",argv[0]);return 1;}
    char *h_enc=NULL; int n=0;
    if(read_passwords(argv[1],&h_enc,&n)!=0) return 1;
    printf("[INFO] %d passwords loaded\n",n);
    size_t bytes=(size_t)n*MAX_PASS_LEN;
    char *h_dec=(char*)calloc(bytes,1);
    char *d_enc=NULL,*d_dec=NULL;
    CUDA_CHECK(cudaMalloc((void**)&d_enc,bytes));
    CUDA_CHECK(cudaMalloc((void**)&d_dec,bytes));
    CUDA_CHECK(cudaMemcpy(d_enc,h_enc,bytes,cudaMemcpyHostToDevice));
    int threads=THREADS_PER_BLOCK;
    int blocks=(n+threads-1)/threads;
    if(blocks<1)blocks=1;
    printf("[INFO] Launch: %d blocks x %d threads\n",blocks,threads);
    decrypt_kernel<<<blocks,threads>>>(d_enc,d_dec,n);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());
    CUDA_CHECK(cudaMemcpy(h_dec,d_dec,bytes,cudaMemcpyDeviceToHost));
    FILE*out=fopen("decrypted.txt","w");
    if(out){for(int i=0;i<n;i++) fprintf(out,"%s\n",h_dec+i*MAX_PASS_LEN);fclose(out);}
    printf("[INFO] Decrypted passwords -> decrypted.txt\n");
    CUDA_CHECK(cudaFree(d_enc)); CUDA_CHECK(cudaFree(d_dec));
    free(h_enc); free(h_dec);
    printf("[INFO] Done.\n");
    return 0;
}

## Step 3 — Generate encrypted passwords using Python

In [ ]:
def encrypt_password(plaintext: str) -> str:
    """Mirror of the C encryption: Caesar +3 then XOR with (i+1)"""
    result = []
    for i, ch in enumerate(plaintext):
        c = (ord(ch) + 3) & 0xFF          # Caesar +3
        c = c ^ (i + 1)                    # XOR position
        result.append(chr(c))
    return ''.join(result)

def decrypt_password(cipher: str) -> str:
    """Reverse of encrypt — for verification"""
    result = []
    for i, ch in enumerate(cipher):
        c = ord(ch) ^ (i + 1)             # undo XOR
        c = (c - 3) & 0xFF                # undo Caesar
        result.append(chr(c))
    return ''.join(result)

plaintext_passwords = [
    'password', 'hello123', 'secure', 'dragon', 'qwerty',
    'letmein', 'sunshine', 'monkey', 'shadow', 'master',
    'football', 'welcome', 'abc123', 'superman', 'batman',
    'trustno1', 'princess', 'starwars', 'baseball', 'access',
    'computing', 'parallel', 'threading', 'cuda_gpu', 'kernel',
    'openmp', 'pthread', 'highperf', 'wolverhampton', 'cs6005',
] * 10  # 300 passwords total

with open('encrypted_passwords.txt', 'w') as f:
    for pw in plaintext_passwords:
        enc = encrypt_password(pw)
        f.write(enc + '\n')

with open('plaintext_passwords.txt', 'w') as f:
    for pw in plaintext_passwords:
        f.write(pw + '\n')

print(f'Generated {len(plaintext_passwords)} passwords')
print('Sample plaintext :', plaintext_passwords[:5])
print('Sample encrypted :', [encrypt_password(p) for p in plaintext_passwords[:5]])
print('Verify decrypt   :', [decrypt_password(encrypt_password(p)) for p in plaintext_passwords[:5]])

## Step 4 — Compile

In [ ]:
!nvcc -O2 -o password_crack password_crack.cu && echo '✅ CUDA compilation successful'

## Step 5 — Run

In [ ]:
!./password_crack encrypted_passwords.txt

## Step 6 — Verify results

In [ ]:
with open('decrypted.txt') as f:
    decrypted = [l.strip() for l in f.readlines()]

with open('plaintext_passwords.txt') as f:
    expected = [l.strip() for l in f.readlines()]

n_total   = len(expected)
n_correct = sum(1 for d, e in zip(decrypted, expected) if d == e)
n_wrong   = n_total - n_correct

print(f'Total passwords : {n_total}')
print(f'Correct         : {n_correct}')
print(f'Wrong           : {n_wrong}')
print(f'Accuracy        : {100*n_correct/n_total:.1f}%')

print('\nSample (expected -> decrypted):')
for e, d in zip(expected[:10], decrypted[:10]):
    status = '✅' if e == d else '❌'
    print(f'  {status}  {e!r}  ->  {d!r}')